In [1]:
!pip -q install pypdf faiss-cpu langchain_community langchain_text_splitters langchain_openai langchain_huggingface tavily-python ipython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.5/334.5 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 71.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
import os
import re
import json
from getpass import getpass
from urllib.parse import urlparse
from typing import TypedDict, List, Dict, Any

import pandas as pd
from IPython.display import display, Markdown

from tavily import TavilyClient
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.config import get_stream_writer

### Set configs

In [3]:
MODEL_NAME = "google/gemini-2.5-flash"
SEARCHES_PER_TOPIC = 4
RESULTS_PER_QUERY = 5
MAX_SOURCES = 6
MAX_CHARS_PER_PAGE = 7000

### Define structured output schemas

In [5]:
class SearchPlan(BaseModel):
    refined_topic: str = Field(description="Cleaner and sharper version of the topic")
    sub_queries: List[str] = Field(description="Search-friendly sub-queries for web research")


class URLSelection(BaseModel):
    selected_ids: List[int] = Field(description="1-based row ids of the best sources to keep")
    

class PageSummary(BaseModel):
    summary: str = Field(description="Short summary of the page")
    top_points: List[str] = Field(description="Most useful facts or claims from the page")
    risks: List[str] = Field(description="Possible risks, limitations, uncertainty, or bias from the page")
    

class SourceItem(BaseModel):
    title: str
    url: str
    why_it_matters: str

class ResearchReport(BaseModel):
    topic: str
    executive_summary: str
    top_findings: List[str]
    risks: List[str]
    sources : List[SourceItem]

### Create all helper functions

In [6]:
def clean_text(text, limit=None):
    text = re.sub(r"\s+", " ", str(text)).strip()
    if limit is not None:
        return text[:limit]
    return text

def get_domain(url):
    try:
        return urlparse(url).netloc.replace("www.", "").strip()
    except:
        return ""

def guess_tavily_topic(topic):
    text = topic.lower()
    news_words = [
        "latest", "recent", "current", "today", "news", "update", "updates",
        "this week", "this month", "2025", "2026", "trend", "trends"
    ]
    if any(word in text for word in news_words):
        return "news"
    return "general"

def dedupe_rows(rows):
    best = {}
    for row in rows:
        url = row["url"]
        if url not in best or row["score"] > best[url]["score"]:
            best[url] = row
        
        return sorted(best.values(), key=lambda x: x["score"], reverse=True)

In [7]:
def make_search_df(rows):
    if not rows:
        return pd.DataFrame(columns=["query", "title", "domain", "score", "url", "snippet"])
    df = pd.DataFrame(rows)
    return df[["query", "title", "domain", "score", "url", "snippet"]]

def make_filtered_df(rows):
    if not rows:
        return pd.DataFrame(columns=["title", "domain", "score", "url", "snippet"])
    df = pd.DataFrame(rows)
    return df[["title", "domain", "score", "url", "snippet"]]

def make_summary_df(rows):
    if not rows:
        return pd.DataFrame(columns=["title", "url", "summary", "top_points", "risks"])
    data = []
    for row in rows:
        data.append(
            {
                "title": row["title"],
                "url": row["url"],
                "summary": row["summary"],
                "top_points": " | ".join(row["top_points"]),
                "risks": " | ".join(row["risks"])
            }
        )
    return pd.DataFrame(data)

In [8]:
def build_markdown_report(report):
    lines = []
    lines.append(f"# Research Report: {report.topic}")
    lines.append("")
    lines.append("## Executive Summary")
    lines.append(report.executive_summary)
    lines.append("")
    lines.append("## Top Findings")
    
    for item in report.top_findings:
        lines.append(f"- {item}")
    lines.append("")
    lines.append("## Risks")
    if report.risks:
        for item in report.risks:
            lines.append(f"- {item}")
    else:
        lines.append("- No major risks were identified from the selected sources.")
    lines.append("")
    lines.append("## Sources")
    
    for i, source in enumerate(report.sources, start=1):
        lines.append(f"{i}. [{source.title}]({source.url}) — {source.why_it_matters}")
    return "\n".join(lines)

### Define workflow state

In [9]:
class ResearchState(TypedDict):
    topic: str
    tavily_topic: str
    sub_queries: List[str]
    search_rows = List[Dict[str, Any]]
    filtered_rows: List[Dict[str, Any]]
    extracted_rows: List[Dict[str, Any]]
    page_summaries: List[Dict[str, Any]]
    final_report : Dict[str, Any]
    report_markdown = str

### Create graph nodes

In [10]:
def generate_queries(state: ResearchState):
    writer = get_stream_writer()
    writer({
        "type": "status", "step": "generate_queries", "message": "Planning search queries..."
    })
    
    planner = llm.with_structured_output(SearchPlan)
    prompt = f"""
        You are planning web research for a short report.

        Topic:
        {state["topic"]}

        Generate a clean refined topic and {SEARCHES_PER_TOPIC} strong search queries.
        """
    plan = planner.invoke(prompt)
    
    seen = set()
    sub_queries = []
    
    for q in plan.sub_queries:
        q = clean_text(q)
        if q and q.lower() not in seen:
            seen.add(q.lower())
            sub_queries.append(q)
        
    sub_queries = sub_queries[:SEARCHES_PER_TOPIC]
    writer({"type": "status", "step": "generate_queries", "message": f"Built {len(sub_queries)} search queries."})
    
    return {
        "tavily_topic" : guess_tavily_topic(state["topic"]),
        "sub_queries" : sub_queries
    }

_search web_

In [ ]:
def search_web(state: ResearchState):
    writer = get_stream_writer()
    writer({"type": "status", "step": "search_web", "message": "Searching the web..."})
    
    rows = []
    for query in state["sub_queries"]:
        writer({"type": "status", "step": "search_web", "message": f"Running query: {query}"})
        
        response = tavily_client.search(
            query=query,
            topic=state["tavily_topic"],
            search_depth="advanced",
            max_results=RESULTS_PER_QUERY,
            include_answer=False,
            include_raw_content=False
        )
        
        for item in response.get("results", []):
            url = item.get("url", "")
            rows.append(
                {
                    "query" : query,
                    "title" : clean_text(item.get("title", "")),
                    "url" : url,
                    "domain" : get_domain(url),
                    "score" : float(item.get("score", 0)),
                    "snippet": clean_text(item.get("content", ""))
                }
            )
    writer({"type": "status", "step": "search_web", "message": f"Collected {len(rows)} candidate results."})
    
    return {"search_rows": rows}

_Filter results_

In [11]:
def filter_results(state: ResearchState):
    deduped = dedupe_rows(state["search_rows"])
    
    if not deduped:
        return {
            "filtered_rows" : []
        }
    candidates = []
    for i, row in enumerate(deduped[:12], start=1):
        candidates.append(
            f"""[{i}]
            Title: {row["title"]}
            Domain: {row["domain"]}
            URL: {row["url"]}
            Score: {row["score"]}
            Snippet: {row["snippet"]}"
            """
        )
    
    selector = llm.with_structured_output(URLSelection)
    prompt = f"""
        You are selecting the best web pages for a short research report.

        Main topic:
        {state["topic"]}

        Pick up to {MAX_SOURCES} rows.
        Choose sources that are:
        - highly relevant
        - diverse
        - credible
        - useful for synthesis
        - not duplicates

        Return only the row ids.

        Candidates:
        {chr(10).join(candidates)}
        """
    
    selected = selector.invoke(prompt)
    filtered_rows = []
    used = set()
    
    for idx in selected.selected_ids:
        if 1 <= idx <= min(len(deduped), 12):
            row = deduped[idx - 1]
            if row["url"] not in used:
                used.add(row["url"])
                filtered_rows.append(row)
    
    
    if not filtered_rows:
        filtered_rows = deduped[:MAX_SOURCES]
    
    return {
        "filtered_rows" : filtered_rows[:MAX_SOURCES]
    }

_Extract pages_